# ⚡ التنبؤ بالطلب على الطاقة باستخدام ML
**Energy Demand Forecasting with Machine Learning**

## 🚀 إعداد التشغيل (Colab · Kaggle · محلي)
الخلية الجاية بتثبّت المكتبات الناقصة وتجيب الداتا تلقائياً على Colab/Kaggle.
**محلياً** (بالـ env بتاع المشروع) هي مجرد no-op — تقدر تتجاهلها.

In [1]:
# 🚀 إعداد تلقائي — Colab / Kaggle / محلي (no-op محلياً)
import os, sys, subprocess, importlib.util
REPO    = "https://github.com/ahmedelgendy11/AI-and-data-science-portfolio"
PROJECT = "ml/b8_demand_forecast"          # مسار المشروع داخل portfolio/
PKGS    = ["xgboost"]          # مكتبات المشروع (تتثبّت لو ناقصة)
for _pkg in PKGS:
    if importlib.util.find_spec(_pkg.replace("-", "_")) is None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", _pkg])
if not os.path.isdir("data"):                 # سحابياً: نجيب الريبو ونروح لمجلد المشروع
    _clone = REPO.rstrip("/").split("/")[-1]
    if not os.path.isdir(_clone):
        subprocess.run(["git", "clone", "-q", REPO + ".git"])
    os.chdir(os.path.join(_clone, "portfolio", PROJECT))
print("✓ جاهز —", os.getcwd())

✓ جاهز — D:\dataanalyst\portfolio\ml\b8_demand_forecast


## 📚 قبل ما تبدأ — محتاج تذاكر إيه
| الموضوع | المصدر المقترح |
|---|---|
| Time Series Feature Engineering (lags, rolling) | Hyndman Ch.5 / Géron Ch.15 |
| XGBoost / Gradient Boosting | Géron Ch.7 |
| Time Series Train/Test Split | Hyndman Ch.3 (لا تستخدم random split!) |
| Evaluation: RMSE, MAE, MAPE | Hyndman Ch.5 |

## 🎯 بيُستخدم في إيه (استخدام واقعي)
- **شركات الكهرباء** بتتنبأ بالطلب عشان تخطط الإنتاج وتتجنب الانقطاعات.
- **المصانع** بتستخدم forecasting عشان تحدد كمية المواد الخام المطلوبة.
- **الـ retail** بيتنبأ بالطلب عشان يدير المخزون ويمنع الـ out-of-stock.

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import TimeSeriesSplit, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import root_mean_squared_error, mean_absolute_error, r2_score
import xgboost as xgb
import warnings
warnings.filterwarnings('ignore')
plt.rcParams['figure.dpi'] = 100

## 1️⃣ تحميل البيانات والاستكشاف

In [3]:
df = pd.read_csv("data/energy_consumption_data.csv", parse_dates=["timestamp"])
df = df.sort_values("timestamp").reset_index(drop=True)
print("Shape:", df.shape)
print("Date range:", df["timestamp"].min(), "->", df["timestamp"].max())
print("\nColumns:", list(df.columns))
print(df.describe().round(1))
df.head()

Shape: (2000, 5)
Date range: 2024-01-01 00:00:00 -> 2024-03-24 07:00:00

Columns: ['timestamp', 'consumption_kwh', 'temperature_c', 'humidity_pct', 'is_holiday']
                           timestamp  consumption_kwh  temperature_c  \
count                           2000           2000.0         2000.0   
mean   2024-02-11 15:29:59.999999744            174.2           15.2   
min              2024-01-01 00:00:00             65.3           -1.3   
25%              2024-01-21 19:45:00            119.5            8.4   
50%              2024-02-11 15:30:00            157.8           15.2   
75%              2024-03-03 11:15:00            224.9           21.9   
max              2024-03-24 07:00:00            376.7           30.5   
std                              NaN             64.5            7.4   

       humidity_pct  is_holiday  
count        2000.0      2000.0  
mean           59.9         0.0  
min            33.8         0.0  
25%            50.2         0.0  
50%            59.8

,timestamp,consumption_kwh,temperature_c,humidity_pct,is_holiday
0,2024-01-01 00:00:00,103.73,15.2,63.9,0
1,2024-01-01 01:00:00,147.99,22.0,54.4,0
2,2024-01-01 02:00:00,106.65,21.9,47.2,0
3,2024-01-01 03:00:00,90.38,22.2,55.5,1
4,2024-01-01 04:00:00,184.63,25.8,42.1,0


In [4]:
fig, axes = plt.subplots(3, 1, figsize=(14, 9), sharex=True)

axes[0].plot(df["timestamp"], df["consumption_kwh"], linewidth=0.5, color="#2ecc71")
axes[0].set_title("استهلاك الطاقة (kWh)")
axes[0].set_ylabel("kWh")

axes[1].plot(df["timestamp"], df["temperature_c"], linewidth=0.5, color="#e74c3c")
axes[1].set_title("درجة الحرارة")
axes[1].set_ylabel("°C")

axes[2].plot(df["timestamp"], df["humidity_pct"], linewidth=0.5, color="#3498db")
axes[2].set_title("نسبة الرطوبة")
axes[2].set_ylabel("%")

plt.tight_layout()
plt.savefig("data/timeseries_overview.png", dpi=120, bbox_inches="tight")
plt.show()

## 2️⃣ هندسة الميزات الزمنية (Time Series Features)
**القاعدة الذهبية:** في السلاسل الزمنية، الميزات الأهم هي:
1. **Lag features** — قيم الماضي
2. **Rolling statistics** — متوسط/انحراف آخر N ساعة
3. **Calendar features** — الساعة، اليوم، الشهر

In [5]:
df["hour"] = df["timestamp"].dt.hour
df["dayofweek"] = df["timestamp"].dt.dayofweek
df["month"] = df["timestamp"].dt.month
df["is_weekend"] = (df["dayofweek"] >= 5).astype(int)

for lag in [1, 2, 3, 6, 12, 24]:
    df[f"lag_{lag}"] = df["consumption_kwh"].shift(lag)

for win in [3, 6, 12, 24]:
    df[f"rolling_mean_{win}"] = df["consumption_kwh"].shift(1).rolling(win).mean()
    df[f"rolling_std_{win}"] = df["consumption_kwh"].shift(1).rolling(win).std()

df = df.dropna().reset_index(drop=True)
print(f"Shape after features: {df.shape}")
print("New columns:", [c for c in df.columns if c not in ["timestamp", "consumption_kwh", "temperature_c", "humidity_pct", "is_holiday"]])

Shape after features: (1976, 23)
New columns: ['hour', 'dayofweek', 'month', 'is_weekend', 'lag_1', 'lag_2', 'lag_3', 'lag_6', 'lag_12', 'lag_24', 'rolling_mean_3', 'rolling_std_3', 'rolling_mean_6', 'rolling_std_6', 'rolling_mean_12', 'rolling_std_12', 'rolling_mean_24', 'rolling_std_24']


## 3️⃣ تقسيم البيانات (Temporal Split)
**⚠️ مهم جداً:** في السلاسل الزمنية، **ما بنستخدمش** `train_test_split` العشوائي!
بنقسّم **بالترتيب الزمني** — الأقدم للتدريب والأحدث للاختبار.

In [6]:
target = "consumption_kwh"
features = [c for c in df.columns if c not in [target, "timestamp"]]

split_idx = int(len(df) * 0.8)
train = df.iloc[:split_idx]
test = df.iloc[split_idx:]

X_train, y_train = train[features], train[target]
X_test, y_test = test[features], test[target]

print(f"Train: {len(train)} rows ({train['timestamp'].min().date()} -> {train['timestamp'].max().date()})")
print(f"Test:  {len(test)} rows ({test['timestamp'].min().date()} -> {test['timestamp'].max().date()})")

Train: 1580 rows (2024-01-02 -> 2024-03-07)
Test:  396 rows (2024-03-07 -> 2024-03-24)


## 4️⃣ تدريب ومقارنة النماذج

In [7]:
models = {
    "Ridge": Ridge(alpha=1),
    "Random Forest": RandomForestRegressor(n_estimators=200, max_depth=12, random_state=42),
    "XGBoost": xgb.XGBRegressor(n_estimators=300, learning_rate=0.05, max_depth=5,
                                 random_state=42, verbosity=0),
}

results = []
predictions = {}
best_score, best_name = -1, None

for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    predictions[name] = y_pred

    rmse = root_mean_squared_error(y_test, y_pred)
    mae = mean_absolute_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)
    mape = np.mean(np.abs((y_test - y_pred) / y_test)) * 100

    results.append({"Model": name, "RMSE": rmse, "MAE": mae, "R²": r2, "MAPE%": mape})
    if r2 > best_score:
        best_score, best_name = r2, name
    print(f"{name:20s}  RMSE={rmse:.1f}  MAE={mae:.1f}  R²={r2:.4f}  MAPE={mape:.1f}%")

results_df = pd.DataFrame(results).sort_values("R²", ascending=False)
print(f"\n🏆 Best: {best_name} (R²={best_score:.4f})")

Ridge                 RMSE=26.4  MAE=20.3  R²=0.8261  MAPE=12.8%


Random Forest         RMSE=12.9  MAE=9.9  R²=0.9585  MAPE=6.6%


XGBoost               RMSE=12.4  MAE=9.6  R²=0.9617  MAPE=6.4%

🏆 Best: XGBoost (R²=0.9617)


## 5️⃣ تصوير التنبؤات

In [8]:
fig, ax = plt.subplots(figsize=(14, 5))
plot_df = test[["timestamp"]].copy()
plot_df["actual"] = y_test.values
plot_df["predicted"] = predictions[best_name]

ax.plot(plot_df["timestamp"], plot_df["actual"], label="Actual", linewidth=1, alpha=0.8)
ax.plot(plot_df["timestamp"], plot_df["predicted"], label=f"Predicted ({best_name})",
        linewidth=1, alpha=0.8, color="#e74c3c")
ax.set_title("الطلب الفعلي vs التنبؤ")
ax.set_xlabel("Time")
ax.set_ylabel("Consumption (kWh)")
ax.legend()
plt.tight_layout()
plt.savefig("data/forecast_plot.png", dpi=120, bbox_inches="tight")
plt.show()

## 6️⃣ أهمية الميزات

In [9]:
best_model = models[best_name]
if hasattr(best_model, "feature_importances_"):
    fi = pd.Series(best_model.feature_importances_, index=features).sort_values(ascending=True)
    fig, ax = plt.subplots(figsize=(8, 8))
    fi.tail(15).plot.barh(ax=ax, color="#3498db")
    ax.set_title(f"Top 15 Features — {best_name}")
    ax.set_xlabel("Importance")
    plt.tight_layout()
    plt.savefig("data/feature_importance.png", dpi=120, bbox_inches="tight")
    plt.show()

    print("\nTop 5 features:")
    for feat, imp in fi.tail(5).items():
        print(f"  {feat:25s} {imp:.4f}")


Top 5 features:
  rolling_mean_3            0.0463
  temperature_c             0.0500
  dayofweek                 0.0524
  hour                      0.1706
  lag_24                    0.6020


## 7️⃣ التحقق بـ Time Series Cross-Validation
بنستخدم **TimeSeriesSplit** عشان نتأكد إن الأداء مش بالصدفة.

In [10]:
tscv = TimeSeriesSplit(n_splits=5)
best_model_fresh = xgb.XGBRegressor(n_estimators=300, learning_rate=0.05, max_depth=5,
                                     random_state=42, verbosity=0)

X_all, y_all = df[features], df[target]
cv_scores = cross_val_score(best_model_fresh, X_all, y_all, cv=tscv, scoring="r2")

print(f"Time Series CV (5-fold) — {best_name}:")
for i, s in enumerate(cv_scores):
    print(f"  Fold {i+1}: R² = {s:.4f}")
print(f"  Mean: {cv_scores.mean():.4f} ± {cv_scores.std():.4f}")

Time Series CV (5-fold) — XGBoost:
  Fold 1: R² = 0.9443
  Fold 2: R² = 0.9503
  Fold 3: R² = 0.9482
  Fold 4: R² = 0.9482
  Fold 5: R² = 0.9611
  Mean: 0.9504 ± 0.0057


## 8️⃣ الخلاصة

In [11]:
print("=" * 50)
print("⚡ ملخص التنبؤ بالطلب على الطاقة")
print("=" * 50)
print(f"\n🏆 أفضل نموذج: {best_name}")
print(f"   R²: {best_score:.4f}")
best_results = results_df[results_df["Model"] == best_name].iloc[0]
print(f"   RMSE: {best_results['RMSE']:.1f} kWh")
print(f"   MAPE: {best_results['MAPE%']:.1f}%")
print(f"\n📌 أهم الملاحظات:")
print("   1. Lag features (خصوصاً lag_1, lag_24) هي الأهم — الاستهلاك القريب بيتنبأ بالقادم")
print("   2. الساعة من اليوم (hour) مهمة — patterns يومية واضحة")
print("   3. درجة الحرارة بتأثر على الاستهلاك (تكييف/تدفئة)")
print("   4. XGBoost بيتفوق على Linear models لأن العلاقات non-linear")
print("\n✅ التحليل اكتمل بنجاح!")

⚡ ملخص التنبؤ بالطلب على الطاقة

🏆 أفضل نموذج: XGBoost
   R²: 0.9617
   RMSE: 12.4 kWh
   MAPE: 6.4%

📌 أهم الملاحظات:
   1. Lag features (خصوصاً lag_1, lag_24) هي الأهم — الاستهلاك القريب بيتنبأ بالقادم
   2. الساعة من اليوم (hour) مهمة — patterns يومية واضحة
   3. درجة الحرارة بتأثر على الاستهلاك (تكييف/تدفئة)
   4. XGBoost بيتفوق على Linear models لأن العلاقات non-linear

✅ التحليل اكتمل بنجاح!
